In [1]:
import pandas as pd


In [2]:

train_features = pd.read_csv("/kaggle/input/lish-moa/train_features.csv")
train_targets = pd.read_csv("/kaggle/input/lish-moa/train_targets_scored.csv")
train_drugs = pd.read_csv("/kaggle/input/lish-moa/train_drug.csv")
test_features = pd.read_csv("/kaggle/input/lish-moa/test_features.csv")
sample_submission = pd.read_csv("/kaggle/input/lish-moa/sample_submission.csv")


In [3]:
X_train = train_features.merge(train_drugs, on="sig_id", how="left") # extra column  "drug_id"
y_train = train_targets
X_test = test_features

In [4]:
X_train.dtypes

X_train['cp_type'] = X_train['cp_type'].astype('category')
X_train['cp_dose'] = X_train['cp_dose'].astype('category')
X_train['cp_time'] = X_train['cp_time'].astype('category')
drug_counts = X_train['drug_id'].value_counts()
print(drug_counts.head(10))### Define modified dataset for the baseline model

drug_id
cacb2b860    1866
87d714366     718
9f80f3f77     246
8b87a7a83     203
5628cb3ee     202
d08af5d4b     196
292ab2c28     194
d50f18348     186
d1b47f29d     178
67c879e79      19
Name: count, dtype: int64


In [5]:
!pip install iterative-stratification

ERROR: Could not find a version that satisfies the requirement iterative-stratification (from versions: none)
ERROR: No matching distribution found for iterative-stratification


In [6]:
import pandas as pd
import cudf
import cupy as cp
from cuml.ensemble import RandomForestClassifier
from cuml.preprocessing import StandardScaler, OneHotEncoder
from itertools import product
from sklearn.model_selection import GroupKFold, StratifiedKFold
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
import numpy as np
from xgboost import XGBClassifier


train_features = pd.read_csv("/kaggle/input/lish-moa/train_features.csv")
train_targets = pd.read_csv("/kaggle/input/lish-moa/train_targets_scored.csv")
train_drugs = pd.read_csv("/kaggle/input/lish-moa/train_drug.csv")
test_features = pd.read_csv("/kaggle/input/lish-moa/test_features.csv")
sample_submission = pd.read_csv("/kaggle/input/lish-moa/sample_submission.csv")

X_train = train_features.merge(train_drugs, on="sig_id", how="left")
y_train = train_targets
X_test = test_features

X_train['cp_type'] = X_train['cp_type'].astype('category')
X_train['cp_dose'] = X_train['cp_dose'].astype('category')
X_train['cp_time'] = X_train['cp_time'].astype('category')


TARGET_COLS = [col for col in y_train.columns if col != 'sig_id']
ALL_FEATURES = X_train.columns.tolist()
NUM_FEATURES = [col for col in ALL_FEATURES if col.startswith('g-') or col.startswith('c-')]
CAT_FEATURES = ['cp_type', 'cp_time', 'cp_dose']
FEATURES = NUM_FEATURES + CAT_FEATURES


is_compound_train = X_train['cp_type'] != 'ctl_vehicle'
X_train_filtered = X_train[is_compound_train].reset_index(drop=True)
y_train_filtered = y_train[is_compound_train].drop(columns=['sig_id']).reset_index(drop=True)

DRUG_ID = X_train_filtered['drug_id']

X_train_filtered['moa_count'] = y_train_filtered.sum(axis=1)
MOA_COUNT_TARGET = X_train_filtered['moa_count']

X_train_filtered['combined_cv_target'] = DRUG_ID.astype(str) + '_' + MOA_COUNT_TARGET.astype(str)
COMBINED_CV_TARGET = X_train_filtered['combined_cv_target']


X_train_filtered = X_train_filtered.drop(columns=['sig_id', 'moa_count', 'combined_cv_target', 'drug_id'])

N_SPLITS = 5


def make_grouped_multilabel_stratified_folds_gpu(drug_ids, y_labels, n_splits=5, random_state=42):

    if hasattr(y_labels, 'to_pandas'):
        df_y = y_labels.to_pandas()
    else:
        df_y = pd.DataFrame(y_labels) if not isinstance(y_labels, pd.DataFrame) else y_labels.copy()
    
    if hasattr(drug_ids, 'to_pandas'):
        drug_ids_pd = drug_ids.to_pandas()
    else:
        drug_ids_pd = pd.Series(drug_ids) if not isinstance(drug_ids, pd.Series) else drug_ids.copy()
    
    df_y['drug_id'] = drug_ids_pd.values
    y_drug_level = df_y.groupby('drug_id').max().drop(columns=['drug_id'], errors='ignore')
    drug_names = y_drug_level.index.values
    y_drug_matrix = y_drug_level.values

    mskf = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    drug_to_fold = {}

    for fold, (_, val_drug_idx) in enumerate(mskf.split(drug_names, y_drug_matrix)):
        for d in drug_names[val_drug_idx]:
            drug_to_fold[d] = fold

    fold_for_sample = drug_ids_pd.map(drug_to_fold).values
    folds = []

    for fold in range(n_splits):
        val_idx = np.where(fold_for_sample == fold)[0]
        train_idx = np.where(fold_for_sample != fold)[0]
        folds.append((train_idx, val_idx))

    return folds

def multi_label_log_loss_gpu(y_true: cp.ndarray, y_pred: cp.ndarray, epsilon: float = 1e-15) -> cp.ndarray:
    y_pred_clipped = cp.clip(y_pred, epsilon, 1.0 - epsilon)
    
    n_targets = y_true.shape[1]
    target_losses = cp.zeros(n_targets, dtype=cp.float32)
    valid_targets = 0
    
    for i in range(n_targets):
        y_true_target = y_true[:, i]
        y_pred_target = y_pred_clipped[:, i]
        
        unique_classes = cp.unique(y_true_target)
        
        if len(unique_classes) == 1:
            if unique_classes[0] == 1.0:
                loss = -cp.mean(cp.log(y_pred_target))
            else:
                loss = -cp.mean(cp.log(1.0 - y_pred_target))
        else:
            loss = -cp.mean(
                y_true_target * cp.log(y_pred_target) + 
                (1.0 - y_true_target) * cp.log(1.0 - y_pred_target)
            )
        
        if cp.isfinite(loss):
            target_losses[i] = loss
            valid_targets += 1
    
    if valid_targets > 0:
        return cp.mean(target_losses[:valid_targets])
    else:
        return cp.array(float('inf'))

class GPUPreprocessor:
    
    def __init__(self, numeric_features, categorical_features):
        self.numeric_features = numeric_features
        self.categorical_features = categorical_features
        self.scaler = StandardScaler()
        self.encoder = OneHotEncoder(handle_unknown='ignore')
        self.is_fitted = False
        
    def fit(self, X, y=None):
        if self.numeric_features:
            numeric_data = X[self.numeric_features].astype('float32')
            self.scaler.fit(numeric_data)
        
        if self.categorical_features:
            categorical_data = X[self.categorical_features]
            self.encoder.fit(categorical_data)
            
        self.is_fitted = True
        return self
        
    def transform(self, X):
        processed_parts = []
        
        if self.numeric_features:
            X_num = self.scaler.transform(X[self.numeric_features].astype('float32'))
            X_num_array = X_num.to_cupy() if hasattr(X_num, 'to_cupy') else cp.asarray(X_num)
            processed_parts.append(X_num_array)
        
        if self.categorical_features:
            X_cat = self.encoder.transform(X[self.categorical_features])
            if hasattr(X_cat, 'todense'):
                X_cat_array = cp.asarray(X_cat.todense(), dtype=cp.float32)
            elif hasattr(X_cat, 'to_cupy'):
                X_cat_array = X_cat.to_cupy()
            else:
                X_cat_array = cp.asarray(X_cat, dtype=cp.float32)
            processed_parts.append(X_cat_array)
        
        if len(processed_parts) == 1:
            return processed_parts[0]
        else:
            return cp.concatenate(processed_parts, axis=1)
    
    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)

class GPUPipeline:
    def __init__(self, preprocessor, models):
        self.preprocessor = preprocessor
        self.models = models
        self.is_fitted = False
        
    def fit(self, X, y):
        X_processed = self.preprocessor.fit_transform(X)
        
        for i, target_col in enumerate(TARGET_COLS):
            y_target_values = y[target_col].values
            if hasattr(y_target_values, 'to_numpy'):
                y_target_values = y_target_values.to_numpy()
            elif hasattr(y_target_values, 'get'):
                y_target_values = y_target_values.get()
            
            y_target = cp.asarray(y_target_values, dtype=cp.float32)
            self.models[i].fit(X_processed, y_target)
            
        self.is_fitted = True
        return self
        
    def predict_proba(self, X):
        if not self.is_fitted:
            raise ValueError("Pipeline not fitted yet")
            
        X_processed = self.preprocessor.transform(X)
        predictions = cp.zeros((X_processed.shape[0], len(TARGET_COLS)), dtype=cp.float32)
        
        for i, model in enumerate(self.models):
            proba = model.predict_proba(X_processed)
            
            # Convert to cupy array consistently
            if hasattr(proba, 'to_cupy'):
                proba_cp = proba.to_cupy()
            elif hasattr(proba, 'get'):
                proba_cp = cp.asarray(proba.get())
            else:
                proba_cp = cp.asarray(proba)
            
            if proba_cp.shape[1] == 2:
                predictions[:, i] = proba_cp[:, 1]
            else:
                predictions[:, i] = proba_cp.flatten()
                
        return predictions

def run_different_validations_methods_gpu(param_grid, model, model_name):
    
    gkf = GroupKFold(n_splits=N_SPLITS)
    skf_moa = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    skf_combined = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    mskf = MultilabelStratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    

    drug_ids_pd = DRUG_ID.to_pandas() if hasattr(DRUG_ID, 'to_pandas') else DRUG_ID
    y_train_pd = y_train_filtered.to_pandas() if hasattr(y_train_filtered, 'to_pandas') else y_train_filtered
    
    grouped_multilabel_folds = make_grouped_multilabel_stratified_folds_gpu(
        drug_ids_pd, y_train_pd, n_splits=N_SPLITS
    )

    cv_configs = [
        {'name': 'Grouped Multilabel Stratified Folds', 'cv_object': None, 'groups': grouped_multilabel_folds, 'stratify_target': None},
        {'name': 'Drug GroupKFold', 'cv_object': gkf, 'groups': DRUG_ID, 'stratify_target': None},
        {'name': 'MoA StratifiedKFold (Binned)', 'cv_object': skf_moa, 'groups': None, 'stratify_target': MOA_COUNT_TARGET},
        {'name': 'Combined Drug+MoA Stratification', 'cv_object': skf_combined, 'groups': None, 'stratify_target': COMBINED_CV_TARGET},
        {'name': 'Pure Multilabel StratifiedKFold', 'cv_object': mskf, 'groups': None, 'stratify_target': y_train_filtered}
    ]

    best_models = {}

    # Convert to pandas for sklearn compatibility
    X_cpu = X_train_filtered.to_pandas() if hasattr(X_train_filtered, 'to_pandas') else X_train_filtered
    y_cpu = y_train_filtered.to_pandas() if hasattr(y_train_filtered, 'to_pandas') else y_train_filtered
    groups_cpu = DRUG_ID.to_pandas() if hasattr(DRUG_ID, 'to_pandas') else DRUG_ID
    moa_target_cpu = MOA_COUNT_TARGET.to_pandas() if hasattr(MOA_COUNT_TARGET, 'to_pandas') else MOA_COUNT_TARGET
    combined_target_cpu = COMBINED_CV_TARGET.to_pandas() if hasattr(COMBINED_CV_TARGET, 'to_pandas') else COMBINED_CV_TARGET
    y_strat_cpu = y_train_filtered.to_pandas() if hasattr(y_train_filtered, 'to_pandas') else y_train_filtered

    param_names = list(param_grid.keys())
    param_combinations = list(product(*param_grid.values()))

    for config in cv_configs:
        cv_name = config['name']
        cv_object = config['cv_object']
        groups = config['groups']
        stratify_target = config['stratify_target']
    
        current_best_loss = float('inf')
        current_best_params = None
    
        print(f"\nRunning CV: {cv_name}")
    
        for params_tuple in param_combinations:
            current_params = dict(zip(param_names, params_tuple))
            fold_losses = []
        
            
            if cv_name == 'Drug GroupKFold':
                cv_splits = list(cv_object.split(X_cpu, y_cpu, groups=groups_cpu))
            elif cv_name == 'Grouped Multilabel Stratified Folds':
                cv_splits = groups  # This should be the precomputed folds
            elif cv_name == 'MoA StratifiedKFold (Binned)':
                cv_splits = list(cv_object.split(X_cpu, moa_target_cpu))
            elif cv_name == 'Combined Drug+MoA Stratification':
                cv_splits = list(cv_object.split(X_cpu, combined_target_cpu))
            else:
                cv_splits = list(cv_object.split(X_cpu, y_strat_cpu))

            for fold, (train_index, val_index) in enumerate(cv_splits):
                try:

                    
                    train_index = np.asarray(train_index)
                    val_index = np.asarray(val_index)
                    
                   
                    X_train_fold = cudf.DataFrame(X_cpu.iloc[train_index])
                    X_valid_fold = cudf.DataFrame(X_cpu.iloc[val_index])
                    y_train_fold = cudf.DataFrame(y_cpu.iloc[train_index])
                    y_valid_fold = cudf.DataFrame(y_cpu.iloc[val_index])

                    preprocessor = GPUPreprocessor(NUM_FEATURES, CAT_FEATURES)
                    
                    models = []
                    for _ in range(len(TARGET_COLS)):
                        models.append(model(**current_params))
                    
                    current_pipeline = GPUPipeline(preprocessor, models)
                    current_pipeline.fit(X_train_fold, y_train_fold)
                
                    y_valid_pred = current_pipeline.predict_proba(X_valid_fold)
                    
                    
                    y_valid_true_values = y_valid_fold.values
                    if hasattr(y_valid_true_values, 'to_numpy'):
                        y_valid_true_np = y_valid_true_values.to_numpy()
                    elif hasattr(y_valid_true_values, 'get'):
                        y_valid_true_np = y_valid_true_values.get()
                    else:
                        y_valid_true_np = y_valid_true_values
                    
                    y_valid_true = cp.asarray(y_valid_true_np, dtype=cp.float32)
                    
                    val_loss = multi_label_log_loss_gpu(y_valid_true, y_valid_pred)
                    
                    if cp.isnan(val_loss) or cp.isinf(val_loss):
                        fold_losses.append(float('inf'))
                    else:
                        fold_losses.append(float(val_loss))
                        print(f"  Fold {fold + 1}: Loss = {val_loss:.4f}")

                except Exception as e:
                    print(f"  ❌ Fold {fold + 1} failed: {str(e)}")
                    import traceback
                    traceback.print_exc()
                    fold_losses.append(float('inf'))
                    continue

            valid_losses = [loss for loss in fold_losses if loss != float('inf')]
            if valid_losses:
                avg_val_loss = sum(valid_losses) / len(valid_losses)
            else:
                avg_val_loss = float('inf')
        
            print(f"  Params {current_params} | Avg Loss: {avg_val_loss:.4f}")
            
            if avg_val_loss < current_best_loss and avg_val_loss != float('inf'):
                current_best_loss = avg_val_loss
                current_best_params = current_params

        if current_best_params:
            best_models[cv_name] = {
                'best_params': current_best_params,
                'best_loss': current_best_loss
            }
            print(f"Best for {cv_name}: {current_best_loss:.4f} with {current_best_params}")
        else:
            print(f"No valid configuration found for {cv_name}")

   
    final_pipelines = {}
    best_params_all = {}

    X_train_full = cudf.DataFrame(X_cpu)
    y_train_full = cudf.DataFrame(y_cpu)

    print("\nTraining final models on all data...")
    
    for cv_name, result in best_models.items():
        best_params = result['best_params']
    
        print(f"Training final model for {cv_name}...")
    
        models = []
        for _ in range(len(TARGET_COLS)):
            models.append(model(**best_params))
        
        preprocessor = GPUPreprocessor(NUM_FEATURES, CAT_FEATURES)
        final_pipeline = GPUPipeline(preprocessor, models)
        final_pipeline.fit(X_train_full, y_train_full)
    
        final_pipelines[cv_name] = final_pipeline
        best_params_all[cv_name] = best_params
    
        y_train_pred = final_pipeline.predict_proba(X_train_full)
        y_train_true_values = y_train_full.values
        if hasattr(y_train_true_values, 'to_numpy'):
            y_train_true_np = y_train_true_values.to_numpy()
        elif hasattr(y_train_true_values, 'get'):
            y_train_true_np = y_train_true_values.get()
        else:
            y_train_true_np = y_train_true_values
        y_train_true = cp.asarray(y_train_true_np, dtype=cp.float32)
        train_loss = multi_label_log_loss_gpu(y_train_true, y_train_pred)
    
        print(f"Final {cv_name} Train Loss: {train_loss:.4f}")

    # Submission generation
    print("\nGenerating submission files...")

    submission_map = {
        'Drug GroupKFold': f"{model_name}_drug_gkf_submission.csv",
        'MoA StratifiedKFold (Binned)': f"{model_name}_moa_skf_binned_submission.csv",
        'Combined Drug+MoA Stratification': f"{model_name}_combined_strat_submission.csv",
        'Pure Multilabel StratifiedKFold': f"{model_name}_mskf_submission.csv",
        'Grouped Multilabel Stratified Folds': f"{model_name}_grouped_multilabel_folds_submission.csv"
    }

    X_test_pd = X_test.to_pandas() if hasattr(X_test, 'to_pandas') else X_test
    X_test_gpu = cudf.DataFrame(X_test_pd)

    for cv_name, file_name in submission_map.items():
        if cv_name not in final_pipelines:
            continue
            
        print(f"Generating submission for {cv_name}...")

        pipeline = final_pipelines[cv_name]
        y_test_pred = pipeline.predict_proba(X_test_gpu)

        submission = sample_submission.copy()
        submission[TARGET_COLS] = cp.asnumpy(y_test_pred)
        submission.to_csv(file_name, index=False)
        print(f"Saved: {file_name}")

    return final_pipelines, best_models



ModuleNotFoundError: No module named 'iterstrat'

In [ ]:

param_grid = {
    'n_estimators': [50],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.7]
}

model_name = "XGBoost"

#
final_pipelines, best_models = run_different_validations_methods_gpu(
    param_grid,
    model=lambda **kwargs: XGBClassifier(
        random_state=42,
        tree_method='hist',
        device='cuda',
        **kwargs
    ),
    model_name=model_name
)

In [ ]:
print("best_models: Dictionary containing best parameters and loss for each CV method")
print("=" * 80)

for cv_name, result in best_models.items():
    print(f"\nCV Method: {cv_name}")
    print(f"Best Loss: {result['best_loss']:.4f}")
    print("Best Parameters:")
    for param_name, param_value in result['best_params'].items():
        print(f"  {param_name}: {param_value}")
    print("-" * 50)

In [ ]:
param_grid = {
    'max_depth': [5, 10, 15],
    'min_samples_leaf': [5, 10]
}

model_name = "decisionTree_gpu"

final_pipelines, best_models = run_different_validations_methods_gpu(
    param_grid,
    model=lambda **kwargs: RandomForestClassifier(
        n_estimators=1,
        bootstrap=False,
        max_features=1.0,
        n_streams=1,
        **kwargs
    ),
    model_name=model_name
)

In [ ]:
print("best_models: Dictionary containing best parameters and loss for each CV method")
print("=" * 80)

for cv_name, result in best_models.items():
    print(f"\nCV Method: {cv_name}")
    print(f"Best Loss: {result['best_loss']:.4f}")
    print("Best Parameters:")
    for param_name, param_value in result['best_params'].items():
        print(f"  {param_name}: {param_value}")
    print("-" * 50)

In [ ]:
param_grid = {
    'n_estimators': [50],
    'max_depth': [5, 10, 15],
    'max_features': ['sqrt', 0.5]
}


model_name = "randomForest"


final_pipelines, best_models = run_different_validations_methods_gpu(
    param_grid,
    model=lambda **kwargs: RandomForestClassifier(
        random_state=42,
        **kwargs,
        n_streams =5
    ),
    model_name=model_name
)

In [ ]:
print("best_models: Dictionary containing best parameters and loss for each CV method")
print("=" * 80)

for cv_name, result in best_models.items():
    print(f"\nCV Method: {cv_name}")
    print(f"Best Loss: {result['best_loss']:.4f}")
    print("Best Parameters:")
    for param_name, param_value in result['best_params'].items():
        print(f"  {param_name}: {param_value}")
    print("-" * 50)

# Adversarial Validation

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

The Performance of this model will be indicator of how big the problem is. In order to make it work we need to use the model of the same complexity for both the main problem and adversarial validation problem.

## Preprocess

In [ ]:
def prepare_adversarial_data(X_train_orig, X_test_orig, numeric_features, categorical_features):

    X_adv = pd.concat([X_train_orig, X_test_orig], axis=0).reset_index(drop=True)
    y_adv = np.array([0] * len(X_train_orig) + [1] * len(X_test_orig))
    
    return X_adv, y_adv

## ROC AUC Score

In [ ]:

best_params_hardcode = {
    'n_estimators': 50,
    'max_depth': 3,
    'learning_rate': 0.1,
    'subsample': 0.7
}


# the same as was in GPUPipeline
X_train_orig = X_train_filtered.to_pandas() if hasattr(X_train_filtered, 'to_pandas') else X_train_filtered.copy()
X_test_orig = X_test.to_pandas() if hasattr(X_test, 'to_pandas') else X_test.copy()

if 'sig_id' in X_test_orig.columns:
    X_test_orig = X_test_orig.drop(columns=['sig_id'])

X_adv, y_adv = prepare_adversarial_data(X_train_orig, X_test_orig, NUM_FEATURES, CAT_FEATURES)

preprocessor = GPUPreprocessor(NUM_FEATURES, CAT_FEATURES)
X_adv_processed = preprocessor.fit_transform(cudf.DataFrame(X_adv))
X_adv_processed_cpu = cp.asnumpy(X_adv_processed)



X_adv_train, X_adv_val, y_adv_train, y_adv_val = train_test_split(
    X_adv_processed_cpu, y_adv, test_size=0.3, stratify=y_adv, random_state=42
)


print("\nTraining adversarial validation model...")
adv_model = XGBClassifier(
    random_state=42,
    tree_method='hist',
    device='cuda',
    **best_params_hardcode
)

adv_model.fit(X_adv_train, y_adv_train)
y_adv_pred_proba = adv_model.predict_proba(X_adv_val)

roc_auc = roc_auc_score(y_adv_val, y_adv_pred_proba[:, 1])

print(f"\nAdversarial Validation Results:")
print(f"ROC AUC Score: {roc_auc:.4f}")





## Plot ROC curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_adv_val, y_adv_pred_proba[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Adversarial Validation - ROC Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:


import os

print(os.listdir("/kaggle/input"))
import shutil

shutil.copy("/kaggle/input/xgboost-mskf-submission/submission.csv", "/kaggle/working/submission.csv")

